In [3]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine( 
    # ESCRITORIO:
    # "mssql+pyodbc://FERCHUSERVER/Northwind?driver=SQL+Server&trusted_connection=yes"
    # NOTEBOOK:
     "mssql+pyodbc://.\\SQLEXPRESS/Northwind?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"

)

"""
Pedido del cliente: "Quiero ver por categoría y trimestre, tanto el 
total facturado como el ticket promedio por pedido, en la misma tabla."

"""

# tablas:
ca=pd.read_sql("select CategoryID, CategoryName from Categories", engine)
p=pd.read_sql("select ProductID, ProductName, CategoryID from Products", engine)
o=pd.read_sql("select OrderID, OrderDate from Orders", engine)
od=pd.read_sql("select OrderID, ProductID, Quantity, UnitPrice, Discount from [Order Details]", engine)

# merge:
ca_p=pd.merge(ca, p, on="CategoryID")
cap_od=pd.merge(ca_p, od, on="ProductID")
df=pd.merge(cap_od, o, on="OrderID")

# crear monto:
df["monto"]=df["Quantity"] * df["UnitPrice"] * (1 - df["Discount"])

# crear columna de trimestre:
df["trimestre"]=df["OrderDate"].dt.quarter


# pivot:
informe=pd.pivot_table(
    df,
    index=["CategoryID","CategoryName"],
    columns="trimestre",
    values="monto",
    aggfunc=["sum","mean"],
    fill_value=0
)
print(informe)
# python repaso2.py

                                     sum                              \
trimestre                              1             2             3   
CategoryID CategoryName                                                
1          Beverages       124993.004896  52400.774942  32590.629980   
2          Condiments       34831.204956  23825.669970  19303.437481   
3          Confections      63102.711431  32631.450483  37395.742446   
4          Dairy Products   67211.314956  59974.994927  46091.839947   
5          Grains/Cereals   31817.337479  25114.104975  17591.184989   
6          Meat/Poultry     48331.531461  37236.859968  31088.889985   
7          Produce          24791.319994  30245.192463  14599.194990   
8          Seafood          41702.052447  23880.309961  35005.419971   

                                               mean                          \
trimestre                             4           1           2           3   
CategoryID CategoryName                          